In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# =========================
# 1. Load data
# =========================
df = pd.read_csv('../data/RawData/telco_customer_data_v2.csv')

In [3]:
# =========================
# 2. Basic overview
# =========================
print("Dataset shape:")
print(df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

Dataset shape:
(70000, 21)

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:
customerID           object
gender               object
SeniorCitizen        object
Partner              object
Dependents           object
tenure              float64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Missing values:
cu

In [4]:

# =========================
# 3. Standardize raw text values
# =========================
# Strip spaces from column names
df.columns = df.columns.str.strip()


# Strip spaces inside object columns
object_cols = df.select_dtypes(include="object").columns
for col in object_cols:
    df[col] = df[col].str.strip()

# Replace common fake-missing values with np.nan
missing_like_values = ["", " ", "NA", "N/A", "na", "n/a", "nan","null", "NULL", "None", "none", "unknown", "Unknown"]
df.replace(missing_like_values, np.nan, inplace=True)


In [6]:

# =========================
# 4. Missing values report
# =========================
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values(by="missing_count", ascending=False)

print("\nMissing values report:")
print(missing_report)




Missing values report:
                  missing_count  missing_percent
PaymentMethod              3569             5.10
Dependents                 3565             5.09
Partner                    3530             5.04
OnlineSecurity             2922             4.17
DeviceProtection           2894             4.13
StreamingTV                2827             4.04
StreamingMovies            2785             3.98
OnlineBackup               2747             3.92
TechSupport                2733             3.90
MultipleLines              1868             2.67
TotalCharges               1169             1.67
gender                      748             1.07
SeniorCitizen               659             0.94
tenure                      567             0.81
MonthlyCharges              388             0.55
Churn                        54             0.08
customerID                    0             0.00
PhoneService                  0             0.00
InternetService               0             0

In [7]:
# =========================
# 5. Duplicate checks
# =========================
full_duplicates = df.duplicated().sum()
customer_id_duplicates = df["customerID"].duplicated().sum() if "customerID" in df.columns else None

print("\nDuplicate rows:", full_duplicates)
print("Duplicate customerID values:", customer_id_duplicates)



Duplicate rows: 0
Duplicate customerID values: 0


In [10]:
numeric_object_cols = []

for col in df.columns:
    if df[col].dtype == "object":
        converted = pd.to_numeric(df[col], errors="coerce")
        numeric_ratio = converted.notna().mean()

        if numeric_ratio > 0.7:
            numeric_object_cols.append(col)

print("Columns likely numeric but stored as object:")
print(numeric_object_cols)

Columns likely numeric but stored as object:
['SeniorCitizen', 'TotalCharges']


In [11]:
# =========================
# 7. Convert expected numeric columns
# =========================

for col in numeric_object_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nData types after numeric conversion:")
print(df[numeric_object_cols].dtypes)



Data types after numeric conversion:
SeniorCitizen    float64
TotalCharges     float64
dtype: object


In [15]:
# =========================
# 8. Negative value checks
# =========================
numeric_cols = df.select_dtypes(include=[np.number]).columns

negative_report = {}
for col in numeric_cols:
    negative_count = (df[col] < 0).sum()
    negative_report[col] = negative_count

negative_report_df = pd.DataFrame.from_dict(
    negative_report, orient="index", columns=["negative_count"]
).sort_values(by="negative_count", ascending=False)

print("\nNegative values report:")
print(negative_report_df)




Negative values report:
                negative_count
tenure                    1162
TotalCharges               198
SeniorCitizen                0
MonthlyCharges               0


In [16]:

# =========================
# 9. Zero value checks for important columns
# =========================

numeric_cols = df.select_dtypes(include=[np.number]).columns

zero_report = {}
for col in numeric_cols:
    if col in df.columns:
        zero_count = (df[col] == 0).sum()
        zero_report[col] = zero_count

zero_report_df = pd.DataFrame.from_dict(
    zero_report, orient="index", columns=["zero_count"]
)

print("\nZero values report:")
print(zero_report_df)



Zero values report:
                zero_count
SeniorCitizen        52405
tenure                   0
MonthlyCharges           0
TotalCharges          1262


In [17]:
# =========================
# 10. Categorical consistency checks
# =========================
print("\nCategorical value inspection:")

for col in object_cols:
    if col in df.columns:
        print(f"\nColumn: {col}")
        print(df[col].value_counts(dropna=False).head(15))




Categorical value inspection:

Column: customerID
customerID
CUST00001    1
CUST00002    1
CUST00003    1
CUST00004    1
CUST00005    1
CUST00006    1
CUST00007    1
CUST00008    1
CUST00009    1
CUST00010    1
CUST00011    1
CUST00012    1
CUST00013    1
CUST00014    1
CUST00015    1
Name: count, dtype: int64

Column: gender
gender
Female    34110
Male      33252
NaN         748
Man         500
male        374
FEMALE      351
m           339
f           326
Name: count, dtype: int64

Column: SeniorCitizen
SeniorCitizen
0.0    52405
1.0    10388
NaN     7207
Name: count, dtype: int64

Column: Partner
Partner
No     38344
Yes    28126
NaN     3530
Name: count, dtype: int64

Column: Dependents
Dependents
No     45375
Yes    21060
NaN     3565
Name: count, dtype: int64

Column: PhoneService
PhoneService
Yes    63011
No      6989
Name: count, dtype: int64

Column: MultipleLines
MultipleLines
Yes                 34681
No                  25192
No phone service     8259
NaN                 

In [ ]:
# =========================
# 11. Expected domain checks ///////////////// check this against data dictionary if available
# =========================
expected_categories = {
    "gender": {"Male", "Female"},
    "SeniorCitizen": {"0", "1", 0, 1},
    "Partner": {"Yes", "No"},
    "Dependents": {"Yes", "No"},
    "PhoneService": {"Yes", "No"},
    "MultipleLines": {"Yes", "No", "No phone service"},
    "InternetService": {"DSL", "Fiber optic", "No"},
    "OnlineSecurity": {"Yes", "No", "No internet service"},
    "OnlineBackup": {"Yes", "No", "No internet service"},
    "DeviceProtection": {"Yes", "No", "No internet service"},
    "TechSupport": {"Yes", "No", "No internet service"},
    "StreamingTV": {"Yes", "No", "No internet service"},
    "StreamingMovies": {"Yes", "No", "No internet service"},
    "Contract": {"Month-to-month", "One year", "Two year"},
    "PaperlessBilling": {"Yes", "No"},
    "Churn": {"Yes", "No"}
}

print("\nInvalid category checks:")

for col, valid_values in expected_categories.items():
    if col in df.columns:
        actual_values = set(df[col].dropna().unique())
        invalid_values = actual_values - valid_values
        print(f"{col}: {invalid_values if invalid_values else 'No invalid values found'}")



Invalid category checks:
gender: {'m', 'male', 'f', 'Man', 'FEMALE'}
SeniorCitizen: {'Yes', 'No', 'not senior'}
Partner: No invalid values found
Dependents: No invalid values found
PhoneService: No invalid values found
MultipleLines: No invalid values found
InternetService: No invalid values found
OnlineSecurity: {'Y', 'True'}
OnlineBackup: {'Y', 'True'}
DeviceProtection: {'Y', 'True'}
TechSupport: {'Y', 'True'}
StreamingTV: {'Y', 'True'}
StreamingMovies: {'Y', 'True'}
Contract: {'M-M'}
PaperlessBilling: No invalid values found
Churn: {'CHURNED', 'NO CHURN', 'Y', 'N'}


In [ ]:
# =========================
# 12. Rows with major issues ////////////// check this against data dictionary if available
# =========================
issue_flags = pd.DataFrame(index=df.index)

issue_flags["missing_any"] = df.isna().any(axis=1)
issue_flags["duplicate_customerID"] = df["customerID"].duplicated(keep=False) if "customerID" in df.columns else False
issue_flags["negative_tenure"] = df["tenure"] < 0 if "tenure" in df.columns else False
issue_flags["negative_monthly"] = df["MonthlyCharges"] < 0 if "MonthlyCharges" in df.columns else False
issue_flags["negative_total"] = df["TotalCharges"] < 0 if "TotalCharges" in df.columns else False

issue_flags["has_issue"] = issue_flags.any(axis=1)

problem_rows = df[issue_flags["has_issue"]].copy()

print("\nNumber of rows with at least one issue:", problem_rows.shape[0])



Number of rows with at least one issue: 32711


In [19]:
# =========================
# 13. Final data quality summary table
# =========================
summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean() * 100).round(2).values,
    "unique_values": df.nunique(dropna=True).values
})

print("\nFinal data quality summary:")
print(summary.sort_values(by="missing_count", ascending=False))



Final data quality summary:
              column    dtype  missing_count  missing_percent  unique_values
2      SeniorCitizen  float64           7207            10.30              2
19      TotalCharges  float64           4474             6.39          48189
17     PaymentMethod   object           3569             5.10              5
4         Dependents   object           3565             5.09              2
3            Partner   object           3530             5.04              2
9     OnlineSecurity   object           2922             4.17              5
11  DeviceProtection   object           2894             4.13              5
13       StreamingTV   object           2827             4.04              5
14   StreamingMovies   object           2785             3.98              5
10      OnlineBackup   object           2747             3.92              5
12       TechSupport   object           2733             3.90              5
7      MultipleLines   object           1868   

In [17]:
# =========================
# 14. Optional export
# =========================
print("Data understanding phase completed.")

Data understanding phase completed.
